# 01. 정상 GPS L1 C/A IQ 생성 및 기초 검증

## 연구 목적
정상 시나리오의 재현 정보를 확인하고, raw IQ의 시간파형·I/Q 분포·PSD·스펙트로그램을 단계별로 관찰합니다.

## 입력
- `configs/gps_l1ca_static.example.yaml`
- `artifacts/rf_runs/*/manifest.json`
- `artifacts/rf_runs/*/gps_l1ca_s8_iq.bin`

기존 run이 없다면 먼저 `python scripts/generate_iq.py`를 실행합니다.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("저장소 루트 또는 notebooks/에서 Notebook을 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
ARTIFACTS = PROJECT_ROOT / "artifacts"
print("PROJECT_ROOT:", PROJECT_ROOT)


## 중간 확인 1 — 최신 정상 run과 manifest

In [ ]:
import json
from pprint import pprint
from gnss_doppler_lab.research_sequence import latest_run, load_run_manifest

run_dir = latest_run(ARTIFACTS / "rf_runs")
manifest = load_run_manifest(run_dir)
iq_path = run_dir / "gps_l1ca_s8_iq.bin"
print("RUN:", run_dir.name)
pprint(manifest)

## 중간 확인 2 — IQ 수치와 일부 원시 표본
전체 파일을 화면에 출력하지 않고 분석에 필요한 최대 1초만 읽습니다.

In [ ]:
import numpy as np
from gnss_doppler_lab.iq_visualization import load_s8_iq, summarize_iq

sample_rate_hz = float(manifest.get("iq", {}).get("rf_sample_rate_hz", manifest.get("output", {}).get("rf_sample_rate_hz", 2_600_000)))
iq = load_s8_iq(iq_path, max_complex_samples=int(sample_rate_hz))
summary = summarize_iq(iq, sample_rate_hz=sample_rate_hz)
print(json.dumps(summary, indent=2))
print("첫 10개 복소 표본:", iq[:10])

## 중간 확인 3 — 시간파형과 I/Q 밀도

In [ ]:
import matplotlib.pyplot as plt

n = min(5000, iq.size)
t_ms = np.arange(n) / sample_rate_hz * 1e3
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(t_ms, iq.real[:n], lw=.6, label='I')
ax[0].plot(t_ms, iq.imag[:n], lw=.6, alpha=.8, label='Q')
ax[0].set(title='Time-domain I/Q', xlabel='Time (ms)', ylabel='ADC level')
ax[0].legend(); ax[0].grid(alpha=.25)
stride = max(1, iq.size // 40000)
pts = iq[::stride][:40000]
ax[1].hexbin(pts.real, pts.imag, gridsize=75, mincnt=1, bins='log')
ax[1].set(title='I/Q density', xlabel='I', ylabel='Q'); ax[1].set_aspect('equal')
plt.show()

## 중간 확인 4 — 상대 PSD와 스펙트로그램

In [ ]:
nfft = min(65536, 2 ** int(np.floor(np.log2(iq.size))))
x = iq[:nfft] * np.hanning(nfft)
p = np.fft.fftshift(np.fft.fft(x))
p_db = 20*np.log10(np.maximum(np.abs(p), 1e-12)); p_db -= p_db.max()
f_mhz = np.fft.fftshift(np.fft.fftfreq(nfft, 1/sample_rate_hz))/1e6
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(f_mhz, p_db, lw=.7); ax[0].set_ylim(-100, 5)
ax[0].set(title='Relative baseband spectrum', xlabel='Offset (MHz)', ylabel='dB'); ax[0].grid(alpha=.25)
count = min(iq.size, int(sample_rate_hz*.2))
ax[1].specgram(iq[:count], NFFT=2048, Fs=sample_rate_hz, noverlap=1536, cmap='magma')
ax[1].set(title='Short-time spectrogram', xlabel='Time (s)', ylabel='Frequency (Hz)')
plt.show()

## 판정

- [ ] IQ byte 수가 짝수이고 예상 duration과 일치한다.
- [ ] peak가 signed 8-bit 한계에 붙지 않아 clipping이 없다.
- [ ] 평균 I/Q가 0 근처이고 한 축으로 심하게 치우치지 않는다.
- [ ] 스펙트럼에 비정상적으로 강한 협대역 tone 또는 aliasing이 없다.
- [ ] manifest에 UTC·위치·NAV/IQ 해시·sample rate가 기록돼 있다.

이 단계의 그래프는 **기초 건전성 검사**이며 PRN/Doppler 정확성의 최종 검증은 아닙니다.

## 다음 단계
`02_gnss_sdr_receiver_analysis.ipynb`에서 GNSS-SDR acquisition·tracking·observables를 확인합니다.